In [1]:
# We import mysql.connector as well as the faker library
import mysql.connector as connector
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()

In [2]:
# We establish kwargs for connection
connection=connector.connect(
    database="little_lemon_schema",
    user="little_lemon",
    password="capstone",
    auth_plugin='mysql_native_password'
)

In [ ]:
# Create the cursor
cursor = connection.cursor()
cursor.execute("USE little_lemon_schema")
# Confirm the database in use
print("Database in use is:", connection.database)

Database is use is: little_lemon_schema


In [ ]:
'''To avoid errors on this current given schema, the insertion sequence must be:
Level 1 (Independent): Insert into CustomerDetails, Staff, and MenuItem.
Level 2 (The Order): Insert into Orders. (This works because the Customer and Staff already exist).
Level 3 (The Delivery): Call AddOrderDeliveryStatus. (This works because the OrderID now exists).'''

# A mapping to make the menu look professional
cuisine_dishes = {
    'Italian': [
        ('Lasagna Classic', 'Main'), ('Pesto Gnocchi', 'Main'), ('Tiramisu', 'Dessert'),
        ('Bruschetta', 'Starter'), ('Risotto Milanese', 'Main'), ('Aperol Spritz', 'Drink'),
        ('Margherita Pizza', 'Main'), ('Panna Cotta', 'Dessert'), ('Minestrone', 'Starter')
    ],
    'French': [
        ('Ratatouille', 'Main'), ('Coq au Vin', 'Main'), ('Quiche Lorraine', 'Main'), 
        ('Crème Brûlée', 'Dessert'), ('Escargot', 'Starter'), ('Bouillabaisse', 'Main'),
        ('Onion Soup', 'Starter'), ('Profiteroles', 'Dessert'), ('French 75', 'Drink')
    ],
    'Greek': [
        ('Moussaka', 'Main'), ('Souvlaki', 'Main'), ('Baklava', 'Dessert'),
        ('Dolmades', 'Starter'), ('Spanakopita', 'Starter'), ('Ouzo', 'Drink'),
        ('Greek Salad', 'Starter'), ('Pastitsio', 'Main'), ('Galaktoboureko', 'Dessert')
    ],
    'Mexican': [
        ('Enchiladas Verdes', 'Main'), ('Al Pastor Tacos', 'Main'), ('Churros', 'Dessert'),
        ('Guacamole & Chips', 'Starter'), ('Chiles Rellenos', 'Main'), ('Margarita', 'Drink'),
        ('Pozole', 'Main'), ('Ceviche', 'Starter'), ('Tres Leches Cake', 'Dessert')
    ]
}

# Use a set to store unique (Cuisine, Name, Type) tuples
unique_items = set()

while len(unique_items) < 36:
    cuisine = random.choice(list(cuisine_dishes.keys()))
    dish, dish_type = random.choice(cuisine_dishes[cuisine])
    
    # Add to set (sets automatically ignore duplicates)
    unique_items.add((cuisine, dish, dish_type))

# We now insert the unique items
for m_cuisine, m_name, m_type in unique_items:
    # Basic tiered pricing logic
    if m_type == 'Main':
        m_price = round(random.uniform(25.00, 50.00), 2)
    elif m_type == 'Starter':
        m_price = round(random.uniform(10.00, 20.00), 2)
    else: # Desserts and Drinks
        m_price = round(random.uniform(5.00, 15.00), 2)
        
    m_cost = round(m_price * 0.3, 2)
    cursor.callproc('PopulateMenuItems', [m_name, m_type, m_cuisine, m_price, m_cost])

connection.commit()
print("Menu items inserted via 'PopulateMenuItems' Stored Procedure!")

In [ ]:
# Use a set to track unique emails (prevents duplicate customer profiles)
unique_customers = set()
target_count = 125

while len(unique_customers) < target_count:
    first_name = fake.first_name()
    last_name = fake.last_name()
    # We generate an email based on the name for realism
    email = f"{first_name.lower()}.{last_name.lower()}@{fake.free_email_domain()}"
    phone = fake.phone_number()
    # We store the whole tuple in the set to keep the data grouped
    unique_customers.add((first_name, last_name, phone, email))

for f_name, l_name, ph, em in unique_customers:
    cursor.callproc('PopulateCustomerDetails', [f_name, l_name, ph, em])

connection.commit()
print(f"Successfully inserted {len(unique_customers)} unique customers!")

Successfully inserted 125 unique customers!


In [ ]:
# we define our specific hierarchy
# Format: (Role, Salary)
fixed_roles = [
    ('Manager', 75000.00),
    ('Head Chef', 70000.00)
]

# Roles for the remaining staff
general_roles = [
    ('Waiter', 35000.00),
    ('Assistant Chef', 45000.00),
    ('Receptionist', 38000.00)
]

unique_staff = set()
target_count = 12

# We generate unique names/emails until we hit our target
while len(unique_staff) < target_count:
    f_name = fake.first_name()
    l_name = fake.last_name()
    email = f"{f_name.lower()}.{l_name.lower()}@littlelemon.com" # Professional domain
    unique_staff.add((f_name, l_name, email))

# Convert set to list so we can iterate with an index
staff_list = list(unique_staff)

for i in range(len(staff_list)):
    f_name, l_name, em = staff_list[i]
    
    # We assign roles based on index
    if i < len(fixed_roles):
        # First two get the specific VIP roles
        role, salary = fixed_roles[i]
    else:
        # Everyone else gets a random general role
        role, salary = random.choice(general_roles)
    
    # Call the Stored Procedure
    cursor.callproc('PopulateStaff', [f_name, l_name, em, role, salary])

connection.commit()
print(f"Staff hierarchy established: 1 Manager, 1 Head Chef, and {target_count-2} support staff.")

In [5]:
# we fetch IDs from the CustomerDetails table
cursor.execute("SELECT CustomerID FROM CustomerDetails")
customer_ids = [row[0] for row in cursor.fetchall()]

print(f"Ready to link {len(customer_ids)} customers")

Ready to link 125 customers


In [ ]:
target_bookings = 60
successful_bookings = 0

# We generate 60 unique Bookings
while successful_bookings < target_bookings:
    c_id = random.choice(customer_ids)
    
    # Generate a random date in the last/next 30 days
    random_days = random.randint(-30, 30)
    target_date = datetime.now() + timedelta(days=random_days)
    
    # Split the datetime into the DATE and TIME formats your procedure wants
    b_date = target_date.strftime('%Y-%m-%d')
    b_time = target_date.strftime('%H:%M:%S')
    
    # Random realistic data for the other fields
    res_table = random.randint(1, 15) # Assuming 15 tables
    g_count = random.randint(1, 6)    # 1 to 6 guests per booking

    # We Call 'AddBookings' 
    try:
        cursor.callproc('PopulateBookings', [b_date, b_time, res_table, g_count, c_id])
        # We increment only if the call above succeeded
        successful_bookings += 1
    except connector.Error as err:
        if err.errno == 1062:
            # We don't increment successful_bookings here, so the loop runs again
            print(f"Collision: Table {res_table} taken at {b_time}. Retrying...")
            continue
        else:
            print(f"Unexpected error: {err}")
            break # Break only on fatal errors to avoid infinite loops

connection.commit()
print(f"{len(successful_bookings)} Bookings successfully inserted!")

Collision: Table 3 taken at 20:27:01. Retrying...
Collision: Table 10 taken at 20:27:01. Retrying...
Collision: Table 1 taken at 20:27:01. Retrying...
Collision: Table 3 taken at 20:27:01. Retrying...
60 Bookings successfully inserted!


In [6]:
# We fetch BookingID and CustomerReservedID as a list of tuples
cursor.execute("SELECT BookingID, CustomerReservedID FROM Bookings")
bookings_pool = cursor.fetchall() # List of tuples: (BookingID, CustomerID)

# We filter for Service Staff only (no Managers or Head Chefs)
cursor.execute("SELECT StaffID FROM Staff WHERE Role NOT IN ('Manager', 'Head Chef')")
front_of_house_ids = [row[0] for row in cursor.fetchall()]

cursor.execute("SELECT MenuItemID FROM MenuItem")
menu_item_ids = [row[0] for row in cursor.fetchall()]

print(f"Pool ready: {len(bookings_pool)} bookings available for linking.")

Pool ready: 60 bookings available for linking.


In [ ]:
# Orders loop
table_occupancy = {}
orders_created = 0
target_orders = 200
discount = 0.00
all_possible_tables = list(range(1, 16)) # Tables 1 through 15

print(f"Starting generation of {target_orders} orders...")

while orders_created < target_orders:
    # We select a service staff member and a random date in the last 30 days
    s_id = random.choice(front_of_house_ids)
    o_date = (datetime.now() - timedelta(days=random.randint(0, 30))).strftime('%Y-%m-%d')

    # We assigns a table
    if o_date not in table_occupancy:
        table_occupancy[o_date] = set()

    # This guarantees we check all 15 tables without repeating 'guesses'
    random.shuffle(all_possible_tables)
    current_table = None

    for table in all_possible_tables:
        if table not in table_occupancy[o_date]:
            current_table = table
            break
    
    # If no table was found after checking all 15
    if current_table is None:
        print(f"Restaurant full on {o_date}, skipping order.")
        continue

    # Logic: 70% chance to fulfill an existing reservation
    if random.random() < 0.7 and bookings_pool:
        # If a reservation is chosen, it's removed from the pool so it can't be reused.
        b_id, c_id = random.choice(bookings_pool)
        base_cost = 20.00 # Base fee for reservations
        bookings_pool.remove((b_id, c_id)) 
    else:
        # This branch handles both Walk-ins AND No-Shows
        # We pick any customer from the database for a walk-in order
        b_id = None
        c_id = random.choice(customer_ids)
        base_cost = 0.00 # No base fee for walk-ins
    
    try:
        cursor.callproc('PopulateOrders', [o_date, current_table, base_cost, discount, c_id, b_id, s_id])
        table_occupancy[o_date].add(current_table)
        orders_created += 1
    except connector.Error as err:
        print(f"Transaction failed, retrying: {err}")
        continue

connection.commit()
print(f"Successfully synchronized {orders_created} Orders!")

Starting generation of 200 orders...
Successfully synchronized 200 Orders!


In [ ]:
# We prepare our pools
cursor.execute("SELECT OrderID, BillAmount FROM Orders")
orders_to_process = cursor.fetchall() # List of (OrderID, BaseFee)

cursor.execute("SELECT MenuItemID, Price FROM MenuItem")
menu_items = cursor.fetchall() # List of (MenuItemID, Price)

print(f"Processing {len(orders_to_process)} orders for item assignment...")

for order_id, base_fee in orders_to_process:
    # We simulate a meal: 1 to 4 items per order
    num_items = random.randint(1, 4)
    session_total = float(base_fee) # Start with the $20 or $0 base fee
    
    selected_items = random.sample(menu_items, num_items)
    
    for item_id, price in selected_items:
        # We call the procedure
        quantity = random.randint(1, 5)
        cursor.callproc('AddOrderItems', [order_id, item_id, quantity])
        
        # Add price to the running total
        session_total += float(price)
    
    # 4. UPDATE the original Order with the real total
    update_query = "UPDATE Orders SET BillAmount = %s WHERE OrderID = %s"
    cursor.execute(update_query, (round(session_total, 2), order_id))

connection.commit()
print("All orders updated with actual menu item totals!")

Processing 200 orders for item assignment...
All orders updated with actual menu item totals!


In [ ]:
# 1. Fetch OrderIDs and Dates
cursor.execute("SELECT OrderID, OrderDate FROM Orders")
all_orders = cursor.fetchall()

# Define realistic restaurant delivery statuses
delivery_statuses = ['Pending', 'In Preparation', 'En Route', 'Delivered']

delivery_count = 0
print(f"Checking {len(all_orders)} orders for takeaway status...")

for o_id, o_date in all_orders:
    # 2. Probability check: Only process if it's a takeaway (25% chance)
    if random.random() < 0.25:
        # Assign a random status from our list
        status = random.choice(delivery_statuses)
        
        # Generate a real address for the delivery
        address = fake.address().replace('\n', ', ')
        
        try:
            # 3. Call 'AddOrderDeliveryStatus' procedure
            # (DeliveryDate, DeliveryStatus, DeliveryAddress, OrderID)
            cursor.callproc('PopulateDeliveryStatus', [o_date, status, address, o_id])
            delivery_count += 1
        except connector.Error as err:
            print(f"Error inserting delivery for Order {o_id}: {err}")

connection.commit()
print(f"{delivery_count} Takeaway deliveries registered.")

Checking 200 orders for takeaway status...
Level 4 Complete: 47 Takeaway deliveries registered.
